In [ ]:
# PyPDFLoader 문서들을 다수의 청크로 분할 하는 RecursiveCharacterTextSplitter 청크들을 임베딩 벡터로 변환 시 OpenAI 의 Embedding API를 사용하기 위해 OpenAIEmbeddings,
# 임베딩 벡터들을 적재하기 위한 벡터 데이터베이스인 Chroma 와 Faiss 를 임포트하고, 사용자의 OpenAI API키 값을 현재 실습 환경에 세팅함.
import os
import urllib.request
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma, FAISS

os.environ['OPENAI_API_KEY'] = ''

# 실습에 사용할 2023_북한인권보고서.pdf 파일을 이 책의 코드 저장소로부터 다운로드 함.
#urllib.request.urlretrieve("https://github.com/ai-agent-kr/agent-tutorial/blob/main/Ch02/2023_%EB%B6%81%ED%95%9C%EC%9D%B8%EA%B6%8C%EB%B3%B4%EA%B3%A0%EC%84%9C.pdf", filename='2023_북한인권보고서.pdf')

# 이제 랭체인의 PyPDFLoader()를 통해 PDF파일을 로드함.
# PyPDFLoader(파일명)을 실행해 loader라는 객체를 선언하고, 해당 객체를 통해 load_and_split()을 실행하면 PDF를 여러개의 문서 청크로 분할한 문자열 리스트가 반환됨.
loader = PyPDFLoader("2023_북한인권보고서.pdf")
pages  = loader.load_and_split()
#print('청크의 수 : ', len(pages)) 

# 이 청크들을 ChatGPT 같은 언어 모델들이 처리할 수 있는 적당한 길이로 추가해 분할해 봄.
# RecursiveCharacterTextSplitter()를 이용해 텍스트를 분할하는 text_splitter 객체를 만듬
# 이 때 chunck_size 의 값을 1000으로 지정하면 앞으로 text_splitter로 텍스트로 분할할 때 각 분할된 청크는 길이가 1000을 넘지 않음.
# chunk_overlap 은 텍스트를 분할할 때 각 청크가 내용을 얼만큼 겹치게 할지를 정하는 값으로 0을 사용하면 각 청크의 내용이 겹치지 않음.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

# 문서를 자르는 청킹 전략에서 실습할 때는 파이썬 문자열을 분할하기 위해 create_documents()를 사용했음.
# 하지만 현재는 PyPDFLoader가 로드한 각각의 청크는 파이썬 문자열이 아닌 Document(page_content='내용', metadata={'source'=파일명, 'page': 기존파일에서의 page 번호})와 같은
# 형식을 가진 원소임 문자열이 아닌 위와 같은 형식을 가진 text_splitter로 분할하는 경우 split_documents() 를 사용함.
splitted_docs = text_splitter.split_documents(pages)
#print('분할된 청크의 수 : ', len(splitted_docs))

# 청크의 수가 445개에서 497개로 늘어났음. 실제로 각 청크의 길이를 재보면 1,000 이 넘지 않는 것을 확인할 수 있음.
# 496개의 청크들에 대해 가장 긴 청크의 길이, 가장 짧은 청크의 길이, 청크들의 평균 길이를 구해 봄.
#chunks = [splitted_doc.page_content for splitted_doc in splitted_docs]
#print('청크의 최대 길이:', max(len(chunk) for chunk in chunks))
#print('청크의 최소 길이:', min(len(chunk) for chunk in chunks))
#print('청크의 평균 길이:', sum(map(len, chunks))/len(chunks))

# 이제 496개의 청크를 모두 OpenAI의 Embedding API로 임베딩해서 크로마 데이터베이스에 적재해 봄.
# 각 청크를 임베딩과 동시에 크로마 데이터베이스에 적재할 때는 Chroma.from_documents(청크들의 리스트, OpenAIEmbeddings())를 사용함.
# 뒤에서 실습할 파이스 벡터데이터베이스도 코드 형식이 거의 동일하므로 기억해 둠.
# Chroma.from_documents() 를 통해 벡터 데이터베이스 객체를 만들고 나서 적재된 문서의 수를 출력하는 것은 _collection.count()를 통해 가능함.
#db = Chroma.from_documents(splitted_docs, OpenAIEmbeddings(), persist_directory='./chroma_test.db')
#print('문서의 수 :', db._collection.count())

# 데이터베이스 객체를 만들고 다시 사용자의 입력과 유사도가 높은 문서들을 찾을 때는 similarity_search(사용자 입력)을 사용함.
# 북한 인권 보고서라는 PDF 파일이므로 '북한의 교육 과정'이라는 질의를 입력해 연관 청크들을 찾아봄.
#question = '북한의 교육 과정'
#docs = db.similarity_search(question)
#print('문서의 수 :', len(docs))

# 연관 청크를 4개 찾음. 실제로 출력해 '북한의 교육 과정'과 연관된 문서인지 확인해 봄.
#for doc in docs:
#    print(doc)
#    print("-----"*10)

# 북한의 교육과 관련된 문서 4개가 출력된 것을 확인할 수 있음.
# 크로마 벡터 데이터베이스를 파일로 저장하는 것도 가능함. Chroma.from_documents()에서 persist_directory= '디렉토리명'을 사용함.
# 다음 코드를 실행하면 실제로 코드 실행 경로에 'chroma_test.db'라는 디렉터리가 생김.

# 저장한 데이터베이스 파일을 로드해서 사용해 봄.
#db_from_file = Chroma(persist_directory='./chroma_test.db', embedding_function=OpenAIEmbeddings())
#print('문서의 수 : ', db_from_file._collection.count())

# 앞에서 similarity_search(사용자 입력)을 사용했을 때는 사용자 입력에 대해 유사한 청크 4개를찾아 냈음.
# 내부적으로는 유사도를 구하고 유사도 점수상위4개의 청크를 찾아낸 것임.

# 이번에는 유사한 청크를 상위 3개만 찾도록 강제하고,유사도 점수 또한 출력하도록 해보겠음.
# 그러려면 similarity_search_with_relevance_scores(사용자 입력,k= 찾고자하는 문서 수)와 같이하면 됨.
# 유사한 청크를 상위 3개만 찾도록 강제하기 위해 k값을3으로 지정했고 유사도 점수 상위 3개의 청크를 찾아서 출력함.
#query = '북한의 교육 과정'
#top_three_docs = db_from_file.similarity_search_with_relevance_scores(query, k=3)

#for doc  in top_three_docs:
#    print(doc)
#    print("---"*10)

# 앞서 만든 502 개의 청크들을 모두 OpenAI의 Embedding API로 임베딩해서 파이스 데이터베이스에 적재해 봄.
# 각 청크를 임베딩과 동시에 파이스 데이터베이스에 적재할 때는 FAISS.from_documents(청크들의 리스트, OpenAIEmbeddings())를 사용해 
# 파이스 벡터 데이터베이스 객체인 faiss_db 를 만듬
# 크로마 벡터 데이터베이스를 사용할 때는 코드가 Chroma.from_documents(청크들의 리스트, OpenAIEmbeddings())였던 것과 매우 유사함.
# 하지만 그외 문서의수를 확인하는것, 파일을 저장하고 로드하는 등의일부 코드는 상의함.
faiss_db = FAISS.from_documents(splitted_docs, OpenAIEmbeddings())
print('문서의 수 :', faiss_db.index.ntotal)

# 파이스 벡터 데이터베이스를 파일로 저장하는 것도 가능함. 이를 위해서는 faiss_db.save_local(디렉터리명)을 사용함.
# 다음 코드를 수행하면 실제로 코드 실행 경로에 faiss_index라는 디렉터리가 생김.
# 반대로 FAISS의 load_local(디렉터리명)으로 앞서 저장한 벡터 데이터베이스를 로드할 수 있음.
# 이 때 사용한 임베딩을 인자로 알려줘야 하므로 OpenAIEmbeddings()를 전달함.

# allow_dangerous_deserialization 옵션은 파이썬 객체를 저장하거나 전송할 때 사용하는 파일을 읽을 때 적용됨.
# 일부 파일에 보안위험이 있으면 읽기가 거부되는데 이 옵션은 에러를 발생시키지 않고 해당 파일을 신뢰할 수있으니 무시하고 읽겠다는 의미임.
# 해당 파일은 방금전 사용자가 지정한 것이므로 True로 설정해 무시하고 있음. 파일을다시 읽어서 new_db_faiss라는 벡터 데이터베이스 객체에 저장.
faiss_db.save_local('faiss_index')
new_db_faiss = FAISS.load_local('faiss_index', OpenAIEmbeddings(), allow_dangerous_deserialization=True)

# 크로마와 마찬가지로 '북한의 교육 과정'으로 검색해 연관된 문서를 확인해 봄
question = '북한의 교육 과정'
docs = new_db_faiss.similarity_search(question)


문서의 수 : 497


In [19]:
for doc in docs:
    print(doc)
    print('---'*10)

page_content='2023 북한인권보고서
40
명목의 교육비용이 전가되고 있는 것으로 나타났다. 교과서는 ‘교과
서 요금’이라는 명목으로 일정 금액을 내야하는 경우가 많으며, 교
과서가 모든 학생에게 충분히 제공되지 않고 학년을 마치면 다음 학
년에 교과서를 물려주어야 했다는 사례가 다수 수집되었다. 소학교
부터 학교운영비, 꼬마계획 등의 비용을 내야했다는 진술이 꾸준히 
수집되고 있는데, 학교시설 현대화 작업이 진행되면서 학교꾸리기 
비용이 증가했다고 한다. 학교에서 요구하는 돈이나 물품은 교원에 
의해 사실상 강제되고 있었는데, 비용을 내지 못하는 경우 동급생들 
앞에서 망신을 주거나 비판하여 형편이 어려운 학생들은 학교를 그
만두는 선택을 하는 경우가 많다고 한다. 또한 도시와 농촌 간 교육
환경의 차이가 크며 대학입학에서 출신성분에 의한 차별이 있고, 교
육기회의 제공에도 경제력이 영향을 미치고 있어 성분·지역·경제
력에 따른 차별이 존재하는 것으로 나타났다. 교육환경도 열악한데, 
학교시설의 현대화 작업에도 불구하고 양호실, 도서관, 위생시설이 
없는 학교도 많은 것으로 보인다. 교원에 대한 경제적 보상도 적절
히 이루어지지 않아, 교원들은 생계를 유지하기 위해 잘사는 학부모
의 원조를 받거나 자신의 텃밭에 학생을 동원시키고 있어 학생들은 
제대로 된 교육여건을 보장받지 못하고 있는 것으로 나타났다. 또
한, 일반교육보다 정치사상교육을 앞세우고 있으며 교과과정에 실
탄사격을 하는 군사훈련을 편성하여 학생들을 의무적으로 참석하게 
하고 있다.
북한의 사회보장 제도로는 연로연금, 노동능력상실 연금, 유가족 
연금 등 생계가 결핍된 경우 기초적인 생계를 보장하기 위한 연금제
도가 있으며, 사회보험금의 성격을 지닌 보조금 제도가 있다. 연로' metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2023-07-